<img src="https://raw.githubusercontent.com/IDEALLab/EngiOpt/codex/dcc26-workshop-notebooks/workshops/dcc26/assets/engibench_logo.png" width="560"/>

# Notebook 03 — Writing your own design problem

*A guided tour, not an exercise sheet. Just run the cells top to bottom and read the prose between them.*

> **Colab users:** click **File ➜ Save a copy in Drive** before editing so your changes persist.

## Where we are in the workshop

In **Notebook 00** we walked through the `beams2d` problem and wrote down the eight things any engineering benchmark has to pin down. Notebooks 01 and 02 then consumed that benchmark to train and evaluate a model.

This notebook flips the perspective. Instead of *consuming* a benchmark, we're going to **write one**. That means we, the researcher, have to answer the same eight questions in code — and package the answers so another lab can import our problem with a single line and get the exact same thing we did.

We'll deliberately pick a problem that is **as simple as possible while still being engineering**: no FEM, no images, no meshes. Just a closed-form physics formula you could do on a napkin. That way nothing distracts from the point of this notebook, which is the *contract* — the interface a benchmark has to honour to be reusable.

**You do not need any ML background for this notebook.** Just high-school algebra and a willingness to look at a few lines of Python.

## A tiny engineering problem to anchor the discussion

Imagine a colleague walks into your office and says:

> *"I have a cantilever beam bolted to a wall with a weight hanging off the end. I want the **lightest** rectangular cross-section that doesn't break and doesn't flex more than the spec allows. Can ML help me?"*

Picture it:

```
     wall
      |
      |==================== | ← end of beam
      |         length L    |
                            ↓ P   (tip load)

        cross-section:
              ┌─────────┐
              │         │  height h
              └─────────┘
                 width b
```

We're going to turn that sentence into a proper benchmark problem. The eight things we pinned down in Notebook 00 are exactly what we need to answer now — but this time we're the ones supplying the answers:

1. **What am I allowed to design?** — a cross-section `(h, b)`.
2. **Under what scenarios must it work?** — a tip load `P` and a length `L`.
3. **What makes one design better?** — less **mass**.
4. **When is a candidate invalid?** — if the beam breaks (stress too high) or flexes too much (tip deflection too large).
5. **Is there prior data?** — no, we're the first ones to define this benchmark.
6. **Can I see a design?** — yes, we'll draw the cross-section and annotate its physics.
7. **How do I score a candidate?** — plug `(h, b, P, L)` into three high-school physics formulas.
8. **What do I have to beat?** — a trivial random-search baseline we'll write ourselves.

The rest of this notebook walks through those answers one at a time, turning each one into a piece of Python, and finally bundles them into a single reusable `Problem` class.

### The physics, on a napkin

For a rectangular cantilever of height `h` and width `b`, loaded with force `P` at the tip of a beam of length `L`, three numbers matter (from Euler–Bernoulli beam theory — you do not need to derive these):

- **Max bending stress** at the wall:   `σ = 6 · P · L / (b · h²)`
- **Tip deflection**:                    `δ = 4 · P · L³ / (E · b · h³)`
- **Mass**:                              `m = ρ · L · b · h`

where `E` is Young's modulus and `ρ` is density — both constants of the material (steel).

## Install dependencies (Colab / fresh env only)

Skip this if your local environment already has `engibench` installed.

In [ ]:
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # flip to True to force install locally

if IN_COLAB or FORCE_INSTALL:
    def _pip(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])
    _pip(["engibench[all]", "matplotlib"])
    _pip(["git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt"])
    print("Install complete.")
else:
    print("Using current environment. Set FORCE_INSTALL=True to install here.")

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

import numpy as np
import matplotlib.pyplot as plt
from gymnasium import spaces

from engibench.core import Problem, ObjectiveDirection, OptiStep
from engibench.constraint import bounded, constraint, THEORY

SEED = 7
rng = np.random.default_rng(SEED)

---
## 1 — *What am I allowed to design?*

First we have to describe the **shape of the output**: what, precisely, does a designer (or a model) hand back to us? For this problem a design is just two numbers — the cross-section's height `h` and width `b`, in metres.

EngiBench (like Gymnasium / OpenAI Gym) represents this with a **`Box` space**: a bounded box in ℝⁿ. Every valid design has to live inside this box. Pinning this down matters because the moment you hook up a generative model, its output layer has to target *exactly* this shape and these bounds — no ambiguity allowed.

In [ ]:
# Bounds chosen so "obviously sensible" cross-sections (1–20 cm) all fit.
design_space = spaces.Box(
    low=np.array([0.02, 0.01], dtype=np.float32),  # h_min = 2 cm, b_min = 1 cm
    high=np.array([0.20, 0.10], dtype=np.float32), # h_max = 20 cm, b_max = 10 cm
    dtype=np.float32,
)

print("Design space:", design_space)
print("Shape:       ", design_space.shape, "  (two scalars: [h, b])")
print("Sample:      ", design_space.sample())

**Reading that output.** A design is a **length-2 float array** in fixed bounds. That's the contract: anything — a hand-picked design, a classical optimiser's result, a neural-network output — has to land inside this box. Nothing else counts as a valid submission.

---
## 2 — *Under what conditions must my design work?*

A cantilever beam isn't just *"a beam"*. It's a beam holding **a specific load at a specific length**. Change either one and the right cross-section changes with it.

EngiBench asks us to declare those *operating conditions* up front as a **dataclass** — one field per scenario parameter, with bounds attached via `bounded(...)`. The bounds aren't a suggestion: `check_constraints()` will refuse any scenario that falls outside them, the same way it does for `beams2d`'s `volfrac`.

In [ ]:
@dataclass
class CantileverConditions:
    # Tip load in Newtons (≈ 10 kg to 1 ton hanging off the end).
    load_N: Annotated[float, bounded(lower=100.0, upper=10_000.0)] = 1000.0
    # Beam length in metres.
    length_m: Annotated[float, bounded(lower=0.2, upper=2.0)] = 1.0


cond = CantileverConditions()
print("A default scenario:", cond)

Two fields, two annotated ranges. That's the whole scenario description. Every evaluation from here on will be *relative to a specific `(load_N, length_m)` pair* — never "just a beam".

---
## 3 — *What does "better" actually mean?*

The colleague asked for the **lightest** beam. Mass is a single scalar we want to push **down**. So the objective is one name plus one direction.

In [ ]:
objectives = (("mass_kg", ObjectiveDirection.MINIMIZE),)
print("Objectives:", objectives)

Notice that `mass_kg` is a **physics quantity**, not an ML loss. It has units. Two people comparing methods on this benchmark will always be comparing the same thing, measured in the same unit — which is the whole point of fixing an objective.

---
## 4 — *How do I score a candidate?*

This is the **simulator** — the function that takes `(design, conditions)` and returns the objective value. In the `beams2d` problem this is a real FEM solver that takes seconds. In our toy problem it's **three lines of algebra**, but the *role* it plays is identical: it is the final arbiter of quality, and the thing any method — ML or not — is ultimately judged by.

We'll write it as a plain function first, then fold it into the `Problem` class at the end.

In [ ]:
# Material constants for structural steel — fixed, not designable.
E_PA = 200e9        # Young's modulus [Pa]
RHO_KGM3 = 7850.0   # Density [kg/m^3]


def cantilever_physics(h: float, b: float, load_N: float, length_m: float) -> dict:
    """Closed-form cantilever beam response. Returns stress, deflection, and mass."""
    stress_Pa   = 6.0 * load_N * length_m / (b * h**2)
    deflection_m = 4.0 * load_N * length_m**3 / (E_PA * b * h**3)
    mass_kg     = RHO_KGM3 * length_m * b * h
    return {"stress_Pa": stress_Pa, "deflection_m": deflection_m, "mass_kg": mass_kg}


# Try a hefty cross-section under a 1 kN tip load on a 1 m beam:
r = cantilever_physics(h=0.06, b=0.03, load_N=1000.0, length_m=1.0)
print(f"stress    = {r['stress_Pa']/1e6:7.2f} MPa")
print(f"deflection= {r['deflection_m']*1000:7.2f} mm")
print(f"mass      = {r['mass_kg']:7.2f} kg")

That's our "simulator". It is deterministic, fast, and trivially reproducible — exactly the properties a benchmark's scoring function needs. A bigger benchmark might hide a PDE solver behind this function signature, but the *contract* `(design, scenario) → objective` is the same.

---
## 5 — *When is a candidate design actually invalid?*

Here's where ML reviewers tend to get caught out: a generative model can hit low training loss and still emit designs that are **nonsense** in the physical world. A benchmark has to protect against this with an *independent* validity test that runs regardless of how a design was produced.

For our cantilever, two physical rules must hold:

| Rule | Formula | Category |
|---|---|---|
| Beam must not break under the load | `stress ≤ 250 MPa` (steel yield) | `THEORY` |
| Beam must not flex more than the spec | `deflection ≤ L / 250` (standard serviceability limit) | `THEORY` |

EngiBench calls each of these a **constraint**, written as a tiny function that *asserts* the rule must hold. If the assertion trips, `check_constraints()` collects it as a violation.

In [ ]:
SIGMA_ALLOW_PA = 250e6  # Steel yield stress [Pa]
DEFLECTION_LIMIT_RATIO = 1.0 / 250.0  # Serviceability limit: delta <= L/250


@constraint(categories=THEORY)
def stress_ok(design: np.ndarray, load_N: float, length_m: float, **_) -> None:
    """The beam must not yield under the tip load."""
    h, b = float(design[0]), float(design[1])
    stress = 6.0 * load_N * length_m / (b * h**2)
    assert stress <= SIGMA_ALLOW_PA, (
        f"stress {stress/1e6:.1f} MPa exceeds yield {SIGMA_ALLOW_PA/1e6:.0f} MPa"
    )


@constraint(categories=THEORY)
def deflection_ok(design: np.ndarray, load_N: float, length_m: float, **_) -> None:
    """The tip must not deflect more than L/250."""
    h, b = float(design[0]), float(design[1])
    delta = 4.0 * load_N * length_m**3 / (E_PA * b * h**3)
    limit = length_m * DEFLECTION_LIMIT_RATIO
    assert delta <= limit, (
        f"deflection {delta*1000:.2f} mm exceeds limit {limit*1000:.2f} mm"
    )


print("Constraints defined:", stress_ok.check.__name__, ",", deflection_ok.check.__name__)

Each constraint is just a function that takes the design + the scenario and asserts a physical rule. No ML here — it's a signed-off physics test that any design has to pass, no matter how it was produced.

Crucially, these live **separately from the simulator**. If we only checked the rules inside `simulate()`, a clever-but-wrong model could game the scoring function. An independent validity test is what keeps papers honest.

---
## 6 — *What's the strongest non-ML baseline?*

A benchmark without a reference number to beat is useless: nobody can tell whether a new method is *better* than something that already existed. So before we wrap everything up, we owe the benchmark two more pieces:

- a way to **sample a random design** from the design box (a sensible starting point for anything, and the building block for generating a reference dataset later);
- a **simple optimiser** that takes a starting design and drives it toward lower mass while still obeying the rules.

We'll write both as standalone functions first. The only reason they end up as class methods later is so every `Problem` in EngiBench exposes them under the same names (`random_design`, `optimize`).

### 6.1 A uniform random sample from the design box

This one is almost embarrassingly simple: pick `(h, b)` uniformly inside the bounds we declared in Section 1. That's the whole idea.

In [ ]:
# Three uniform random picks from the design box:
for _ in range(3):
    d = rng.uniform(design_space.low, design_space.high).astype(np.float32)
    print(f"  random design: h = {d[0]*1000:5.1f} mm, b = {d[1]*1000:5.1f} mm")

That's all `random_design()` will do inside the `Problem` class — it just returns a uniform sample from the design box. Most of these won't satisfy our physical rules, and that's fine: *fixing them up* is what the optimiser is for.

### 6.2 A reference optimiser — feasible random-perturbation search

The simplest possible classical optimiser that still respects constraints is:

> *Start somewhere. Nudge the current best design by a small random vector. Throw the nudge out if it breaks a rule. Keep the nudge if it lowers the objective. Repeat.*

No gradients. No solver. Just **feasibility** and **comparison**. Written as a recipe:

1. Start from a design; record its mass if feasible, else `∞`.
2. For each step, sample a Gaussian perturbation of the current best and clip back into the design box.
3. If the candidate violates a constraint, keep the current best and move on.
4. Otherwise score the candidate; if its mass beats the incumbent, adopt it.
5. Append the best-mass-so-far to a history list — that list is the curve you plot.

We first need a tiny helper that asks *"does this design pass both `@constraint` rules?"*. Then the search loop itself is about a dozen lines.

In [ ]:
def is_feasible(design, load_N, length_m):
    """Run both @constraint functions; return True iff neither raises."""
    kwargs = {"design": design, "load_N": load_N, "length_m": length_m}
    return not any(c.check_dict(kwargs) for c in (stress_ok, deflection_ok))


def random_search(start, load_N, length_m, n_steps=500, step_scale=(0.01, 0.005)):
    """Nudge, clip, reject-if-infeasible, keep-if-better. Returns (best_design, mass_curve)."""
    step_scale = np.array(step_scale, dtype=np.float32)
    best = np.clip(start, design_space.low, design_space.high).astype(np.float32)
    if is_feasible(best, load_N, length_m):
        best_mass = cantilever_physics(best[0], best[1], load_N, length_m)["mass_kg"]
    else:
        best_mass = float("inf")
    curve = [best_mass]

    for _ in range(n_steps):
        cand = best + rng.normal(size=2).astype(np.float32) * step_scale
        cand = np.clip(cand, design_space.low, design_space.high)
        if is_feasible(cand, load_N, length_m):
            m = cantilever_physics(cand[0], cand[1], load_N, length_m)["mass_kg"]
            if m < best_mass:
                best, best_mass = cand, m
        curve.append(best_mass)
    return best, curve


start = np.array([0.15, 0.08], dtype=np.float32)   # deliberately heavy starting point
best_standalone, curve = random_search(start, load_N=1000.0, length_m=1.0, n_steps=500)

plt.figure(figsize=(5, 3))
plt.plot(curve)
plt.xlabel("step"); plt.ylabel("best mass so far [kg]")
plt.title("Standalone baseline search"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"Lightest feasible beam found: {curve[-1]:.2f} kg at h={best_standalone[0]*1000:.1f} mm, b={best_standalone[1]*1000:.1f} mm")

A few hundred random nudges already drove the mass from the starting point's ~94 kg down to a much smaller number — not bad for an algorithm with no gradients and roughly a dozen lines of code. That final plateau is our **reference**: any ML method proposing designs for this scenario has to reliably land below it to count as doing something useful.

When we fold this loop into the `Problem` class in the next section, two cosmetic things change — the method is called `optimize()` (to match EngiBench's contract), and each step's history entry is wrapped as an `OptiStep(obj_values=..., step=...)` so other tooling can consume it. Same algorithm, richer wrapper.

---
## 7 — *Putting it all together: a `Problem` class*

We now have every piece we need:

- a `design_space` (what the output looks like)
- a `Conditions` dataclass (what scenario we're evaluating against)
- an `objectives` tuple (the scalar we're pushing on)
- a simulator function (the scoring rule)
- two `@constraint` functions (the validity rules)
- a uniform sampler `random_design` and a random-search baseline `random_search`

The last step is to stitch them into one `Problem` subclass. That's the object other code (and other people) will import. Everything below gets wired into it as class attributes or methods. There's nothing new here — we're just collecting what we already have, under the method names EngiBench expects.

In [ ]:
class CantileverBeamProblem(Problem[np.ndarray]):
    """A minimal analytical benchmark: size a rectangular cantilever for minimum mass."""

    version = 0
    objectives = (("mass_kg", ObjectiveDirection.MINIMIZE),)

    # The scenario (what the designer doesn't get to pick).
    @dataclass
    class Conditions:
        load_N: Annotated[float, bounded(lower=100.0, upper=10_000.0)] = 1000.0
        length_m: Annotated[float, bounded(lower=0.2, upper=2.0)] = 1.0

    # `Config` = the scenario plus any solver-only knobs (we have none).
    @dataclass
    class Config(Conditions):
        max_iter: Annotated[int, bounded(lower=1, upper=10_000)] = 500

    dataset_id = "IDEALLab/cantilever_toy_v0"  # placeholder — no dataset published
    container_id = None

    def __init__(self, seed: int = 0, **kwargs):
        super().__init__(seed=seed)
        self.config = self.Config(**kwargs)
        self.conditions = self.Conditions(
            load_N=self.config.load_N,
            length_m=self.config.length_m,
        )
        self.design_space = spaces.Box(
            low=np.array([0.02, 0.01], dtype=np.float32),
            high=np.array([0.20, 0.10], dtype=np.float32),
            dtype=np.float32,
        )
        self.design_constraints = (stress_ok, deflection_ok)

    # --- The scoring rule ---------------------------------------------------
    def simulate(self, design: np.ndarray, config: dict | None = None) -> np.ndarray:
        cfg = {**self.config.__dict__, **(config or {})}
        h, b = float(design[0]), float(design[1])
        mass = RHO_KGM3 * cfg["length_m"] * b * h
        return np.array([mass], dtype=np.float32)

    # --- A uniformly random sample inside the design box -------------------
    def random_design(self):
        design = self.np_random.uniform(
            self.design_space.low, self.design_space.high
        ).astype(np.float32)
        return design, -1  # -1 = "not sampled from any dataset"

    # --- The classical baseline: simple random-perturbation search --------
    def optimize(self, starting_point, config=None):
        cfg = {**self.config.__dict__, **(config or {})}
        x = np.clip(starting_point, self.design_space.low, self.design_space.high).astype(np.float32)
        best = x.copy()
        best_obj = self.simulate(best, cfg)
        feasible = not self.check_constraints(best, config=cfg)
        best_score = float(best_obj[0]) if feasible else float("inf")
        history = [OptiStep(obj_values=best_obj, step=0)]

        step_scale = np.array([0.01, 0.005], dtype=np.float32)  # 1 cm / 0.5 cm jitter
        for i in range(int(cfg["max_iter"])):
            cand = best + self.np_random.normal(size=best.shape).astype(np.float32) * step_scale
            cand = np.clip(cand, self.design_space.low, self.design_space.high)
            if self.check_constraints(cand, config=cfg):
                history.append(OptiStep(obj_values=best_obj, step=i + 1))
                continue
            cand_obj = self.simulate(cand, cfg)
            if float(cand_obj[0]) < best_score:
                best, best_obj, best_score = cand.copy(), cand_obj, float(cand_obj[0])
            history.append(OptiStep(obj_values=best_obj, step=i + 1))
        return best, history

    # --- How a human looks at a design ------------------------------------
    def render(self, design: np.ndarray, *, open_window: bool = False):
        cfg = self.config
        h, b = float(design[0]), float(design[1])
        phys = cantilever_physics(h, b, cfg.load_N, cfg.length_m)

        fig, ax = plt.subplots(figsize=(4.2, 4.2))
        ax.add_patch(plt.Rectangle((-b/2, 0), b, h, facecolor="#4c78a8", edgecolor="black"))
        pad = 0.12
        ax.set_xlim(-pad, pad); ax.set_ylim(-0.02, 0.22)
        ax.set_aspect("equal"); ax.set_xlabel("width b [m]"); ax.set_ylabel("height h [m]")
        stress_ratio = phys["stress_Pa"] / SIGMA_ALLOW_PA
        delta_limit = cfg.length_m * DEFLECTION_LIMIT_RATIO
        delta_ratio = phys["deflection_m"] / delta_limit
        ax.set_title(
            f"h={h*1000:.1f} mm, b={b*1000:.1f} mm\n"
            f"stress = {phys['stress_Pa']/1e6:.1f} MPa  ({stress_ratio*100:.0f}% of allow)\n"
            f"tip δ  = {phys['deflection_m']*1000:.2f} mm  ({delta_ratio*100:.0f}% of allow)\n"
            f"mass   = {phys['mass_kg']:.2f} kg",
            fontsize=9,
        )
        plt.tight_layout()
        if open_window:
            plt.show()
        return fig, ax


problem = CantileverBeamProblem(seed=SEED)
print("Created problem:", type(problem).__name__)

That single class **is** our benchmark. From this point on, anything that worked on `beams2d` in Notebook 00 works on `problem` — same method names, same return shapes.

Let's prove it by running through the same checklist.

---
## 8 — *Use it exactly like `beams2d`*

### 8.1 Inspect the interface

In [ ]:
print("Design space:  ", problem.design_space)
print("Objectives:    ", problem.objectives)
print("Condition keys:", problem.conditions_keys)
print("Conditions:    ", problem.conditions)

If you squint at those four lines next to the same four lines from Notebook 00, you'll see the same API — we just wrote a different problem behind it.

### 8.2 A feasible design — the scoring function + validity test agree

Let's pick a sensible beefy cross-section, check it obeys the rules, and read off its mass.

In [ ]:
feasible_design = np.array([0.06, 0.03], dtype=np.float32)   # h=6 cm, b=3 cm
scenario = {"load_N": 1000.0, "length_m": 1.0}

violations = problem.check_constraints(feasible_design, config=scenario)
print(f"Checked {violations.n_constraints} constraints — {len(violations)} violated.")
print("Objective:", problem.simulate(feasible_design, config=scenario), "kg")

Zero violations, mass ≈ 14 kg. That's a valid benchmark entry.

### 8.3 An infeasible design — validity test catches it

Now let's try a noticeably *thinner* beam and watch the benchmark reject it.

In [ ]:
thin_design = np.array([0.04, 0.02], dtype=np.float32)  # h=4 cm, b=2 cm
bad = problem.check_constraints(thin_design, config=scenario)
print(f"Violations: {len(bad)} of {bad.n_constraints} constraints.\n")
print(bad)

The thin beam's stress is still OK, but its tip deflection blows past the `L/250` limit — so the benchmark flags it, loudly and independently of whatever objective score we'd compute.

This is the *whole point* of having a separate validity layer: a generative model that happened to produce this design would score it low on mass (hooray!) but the validity check would veto it before it got counted as a win.

### 8.4 A visual

`problem.render(...)` gives us the canonical picture of a design — here, the cross-section annotated with its physics.

In [ ]:
problem.render(feasible_design)
plt.show()

### 8.5 The classical baseline via the class API

We already ran the random-search baseline as a standalone function in Section 6. Let's confirm `problem.optimize(...)` runs the same algorithm through the class and produces a similar curve — just packaged as `OptiStep` records instead of raw floats.

In [ ]:
problem.reset(seed=SEED)
start = np.array([0.15, 0.08], dtype=np.float32)  # start heavy and shrink down
best, history = problem.optimize(start, config={**scenario, "max_iter": 500})

print(f"Optimiser ran for {len(history)} steps")
print(f"Start mass   : {history[0].obj_values[0]:.3f} kg")
print(f"Final mass   : {history[-1].obj_values[0]:.3f} kg")
print(f"Final design : h = {best[0]*1000:.2f} mm, b = {best[1]*1000:.2f} mm")

curve = [s.obj_values[0] for s in history]
plt.figure(figsize=(5, 3))
plt.plot(curve)
plt.xlabel("step"); plt.ylabel("best mass so far [kg]")
plt.title("Baseline optimisation curve"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

A monotonically decreasing curve that plateaus at the lightest feasible cross-section the baseline found. That plateau is the **number to beat**: if your future ML method proposes designs whose mass is reliably below it (and still feasible), that's a meaningful claim. If it isn't, it isn't.

---
## Putting it together

Look back at the eight questions we had to answer. Here's the mapping we just built — but this time the right-hand column is **code we wrote ourselves**:

| Researcher's question | What we wrote | Where it lives on `problem` |
|---|---|---|
| What problem am I on? | `class CantileverBeamProblem(Problem[np.ndarray])` | `type(problem).__name__` |
| What am I designing? | A 2-D `spaces.Box` | `problem.design_space` |
| Under what scenarios? | The `Conditions` dataclass with `bounded()` fields | `problem.conditions` |
| What does better mean? | `objectives = (("mass_kg", MINIMIZE),)` | `problem.objectives` |
| How does this design score? | 3 lines of algebra in `simulate()` | `problem.simulate(design, cfg)` |
| When is a design invalid? | Two `@constraint` functions | `problem.check_constraints(...)` |
| Can I see a design? | `render()` draws the cross-section | `problem.render(design)` |
| What do I have to beat? | A random-search `optimize()` | `problem.optimize(start, cfg)` |

The payoff: **every generative model in EngiOpt** — the CGAN from Notebook 01, the diffusion models, the VAEs — could, in principle, train on `CantileverBeamProblem` **with zero model-code changes**, because they only talk to `problem` through the exact API above. That's what the Problem contract is *for*: decouple the engineering problem from the ML method so the two sides of the research can move independently.

---
## What we deliberately skipped

A real, publishable benchmark needs two more things we didn't do here:

1. **A dataset of reference `(design, scenario, objective)` rows**, large enough for generative models to train on. The standard way to produce one is to run `problem.optimize(...)` thousands of times with scenarios sampled from the condition ranges, and package the results on HuggingFace under the `dataset_id` the class declares.
2. **Metadata for reproducibility** — which version of the physics you used, what units everything is in, which solver settings are baked in, how to cite the dataset.

For a toy problem with a one-line simulator these feel optional. For a real benchmark they're what lets another lab trust your numbers a year later.

---
## Reflect before moving on

1. Which piece of the `Problem` contract felt the most obvious to write? Which felt the least? Why?
2. We put the stress and deflection rules in `@constraint` functions instead of inside `simulate()`. What would go wrong if we merged them into `simulate()` and returned "invalid" as a huge penalty value?
3. Imagine an engineering problem from **your** research — fluid flow, circuits, chemistry, robotics. What would its `design_space`, `Conditions`, objective, and one constraint look like? Sketch them in pseudocode.

## Next

You've now seen the full loop: **consume** a benchmark (Notebook 00), **train against** it (Notebook 01), **evaluate on** it (Notebook 02), and **build** a new one (this notebook). Those four steps are the whole workflow of benchmark-driven research in engineering design — everything else is detail.